# Go2 Track 2 Bonus Project – Fast Policy Edition
**Team: Jiarao_Zhang** | EEC289A/EEC289Q | Target: 200m < 200 s

## Speed math
200m oval: 114.7m curves + 85.3m straights.  
For < 200s at 80% speed on curves: need `VX_MAX ≥ 1.12 m/s`.  
Low-level policy trained to 1.30 m/s; MLP planner commanded up to 1.15 m/s.

## Training strategy
- **Stage 1** (10M steps, vx-only 0–0.9 m/s): builds a stable forward trot foundation
- **Stage 2** (15M steps, vx 0–1.30 m/s + vy ±0.4 + yaw ±1.0): extends speed AND adds turning
  - `command_keep_prob = [0.9, 0.8, 0.8]` — both vy and yaw_rate are actively commanded
  - `tracking_lin_vel: 1.6` + `tracking_ang_vel: 1.2` — reward terms incentivize following velocity commands
  - **Without vy/yaw training the robot cannot turn the oval curves → always fails**

## Why earlier policies failed
| Failure | Root cause |
|---------|-----------|
| v12/v13 fall immediately | stage_1 at 1.2 m/s → no stable gait foundation |
| stage_2 only ran 5M steps | used `runtime_overrides["stage_2_num_timesteps"]` (ignored by train.py) |
| smoke test fall at 2.7s | stage_2 vy/yaw=0, keep_prob=[1,0,0] → robot never learned to turn |

| Cell | Task | Time |
|------|------|------|
| 1 | Mount Drive + Config | 1 min |
| 2 | Install + Clone repos | 5–10 min |
| 3 | Copy Go2 assets | 1 min |
| 4 | Patch planner.py VX_MAX→1.15 | instant |
| 5 | Configure FAST runtime (4 bug fixes) | instant |
| 6 | **Train low-level policy** (stage1 0.9 m/s + stage2 1.30 m/s w/ turning) | **~3-4 h** |
| 7 | Save checkpoint to Drive | instant |
| 8 | Quick smoke test | 3-5 min |
| 9 | **CMA-ES MLP training** (in-process) | **30-60 min** |
| 10 | Save MLP to Drive | instant |
| 11 | Full track eval + video | 5-10 min |
| 12 | submission.json + checklist | instant |
| 13 | Push to GitHub | 1 min |

In [ ]:
# ── CELL 1: Mount Drive + Config ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, subprocess, json, io, tarfile, tempfile, time, urllib.request
from pathlib import Path
from urllib.parse import urlparse

TEAM_NAME            = "Jiarao_Zhang"
COURSE_REPO_URL      = "https://github.com/jiarao76/Final-Project-Track-2-Bonus-Project.git"
COURSE_REPO_BRANCH   = "main"

PLAYGROUND_REPO      = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF       = "dd38c285c6d54266287081e516109f0b15985818"
UNITREE_MUJOCO_REPO  = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF   = "1a37b051a10be723405b7ed6dc839361af036d88"
MENAGERIE_REPO       = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF        = "1b86ece576591213e2b666ebf59508454200ca97"

BASE_DIR             = Path("/content")
COURSE_REPO_DIR      = BASE_DIR / "go2_track_bonus_repo"
PLAYGROUND_DIR       = BASE_DIR / "mujoco_playground"
UNITREE_DIR          = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR        = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

# Checkpoint produced by this run (Cell 6)
CHECKPOINT_DIR       = COURSE_REPO_DIR / "artifacts" / "fast_policy" / "best_checkpoint"

DRIVE_BACKUP         = Path("/content/drive/MyDrive/go2_track_backup")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

print("Drive backup dir :", DRIVE_BACKUP)
print("Checkpoint dir   :", CHECKPOINT_DIR)

def run(cmd):
    cmd = [str(c) for c in cmd]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)

def github_archive_url(repo_url, ref):
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url, ref, target_dir):
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [p for p in tmp_dir.iterdir() if p.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted dir, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def ensure_pinned_repo(repo_url, ref, target_dir):
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
            run(["git", "-C", target_dir, "checkout", ref])
            return
        except subprocess.CalledProcessError:
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)
    try:
        run(["git", "clone", repo_url, target_dir])
        run(["git", "-C", target_dir, "checkout", ref])
    except subprocess.CalledProcessError:
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url, branch, target_dir, reset=False):
    if target_dir.exists():
        if reset:
            shutil.rmtree(target_dir)
        else:
            print(f"+ reuse existing course repo at {target_dir}")
            return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError:
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

print("Cell 1 done.")

In [ ]:
# ── CELL 2: Install packages + Clone repos ────────────────────────────────────
if shutil.which("ffmpeg") is None:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "ffmpeg"])

ensure_pinned_repo(PLAYGROUND_REPO,     PLAYGROUND_REF,     PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_pinned_repo(MENAGERIE_REPO,      MENAGERIE_REF,      MENAGERIE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)

os.chdir(COURSE_REPO_DIR)
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground 2>/dev/null || true
!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}

os.chdir(PLAYGROUND_DIR)
!python -m pip install -q -e .
os.chdir(COURSE_REPO_DIR)

if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))
if str(COURSE_REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(COURSE_REPO_DIR.resolve()))

import jax
import mujoco_playground
print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())

expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError("mujoco_playground imported from wrong location")
print("Cell 2 done.")

In [ ]:
# ── CELL 3: Copy Go2 assets ───────────────────────────────────────────────────
os.chdir(COURSE_REPO_DIR)
!python scripts/copy_go2_assets.py \
    --unitree-dir {UNITREE_DIR} \
    --course-dir {COURSE_REPO_DIR}
print("Assets copied.")

In [ ]:
# ── CELL 4: Patch planner.py – speed limits for sub-200s laps ────────────────
#
# Target: 200m in < 200s  →  avg speed ≥ 1.0 m/s
# Track:  114.7m curves + 85.3m straights
# Assuming 80% speed on curves vs straights:
#   time = 114.7/(0.8*VX_MAX) + 85.3/VX_MAX < 200
#   → VX_MAX > 1.12 m/s
# We set VX_MAX = 1.15 m/s (low-level policy trained to 1.30 m/s).

planner_py = COURSE_REPO_DIR / "track_bonus" / "planner.py"
content = planner_py.read_text()

# The repo has 0.50; patch to 1.15 for sub-200s target
replacements = {
    "_VX_MAX: float = 0.50": "_VX_MAX: float = 1.15",
    "_VX_MAX: float = 0.85": "_VX_MAX: float = 1.15",  # in case already patched
    "_VX_MIN: float = 0.15": "_VX_MIN: float = 0.20",
    "_VY_LIM: float = 0.10": "_VY_LIM: float = 0.25",
    "_VY_LIM: float = 0.20": "_VY_LIM: float = 0.25",  # in case already patched
    "_YAW_LIM: float = 0.30": "_YAW_LIM: float = 0.50",
    "_YAW_LIM: float = 0.45": "_YAW_LIM: float = 0.50",  # in case already patched
}
applied = []
for old, new in replacements.items():
    if old in content:
        content = content.replace(old, new)
        applied.append(f"  {old} → {new}")
planner_py.write_text(content)

print("Patches applied:")
for a in applied:
    print(a)

print("\nCurrent limits:")
for line in content.splitlines():
    if any(k in line for k in ["_VX_MAX", "_VX_MIN", "_VY_LIM", "_YAW_LIM"]):
        print(" ", line.strip())

In [ ]:
# ── CELL 5: Configure FAST runtime ───────────────────────────────────────────
#
# Stage 1: 10M steps, vx-only 0–0.9 m/s, vy/yaw=0
#   → Builds stable forward trot (same approach as HW1 stage_1)
#
# Stage 2: 15M steps, vx 0–1.30 m/s  +  vy ±0.4  +  yaw ±1.0 rad/s
#   → Full locomotion with turning – command_keep_prob allows vy/yaw commands
#   → Proper tracking reward terms to incentivize velocity following
#
# Total: 25M steps (within 30M leaderboard budget)
#
# IMPORTANT FIXES vs previous buggy run:
#   Bug 1: num_timesteps must be set DIRECTLY in stage config
#          (runtime_overrides["stage_N_num_timesteps"] is ignored by train.py)
#   Bug 2: stage_2 command_range MUST include vy and yaw (not just vx)
#          Without this the robot can NEVER learn to turn → fails on oval
#   Bug 3: stage_2 command_keep_prob MUST allow vy/yaw
#          [1,0,0] means vy and yaw are NEVER commanded → no turning
#   Bug 4: stage_2 reward_scales MUST include tracking_lin_vel + tracking_ang_vel
#          Without these terms the policy ignores velocity commands

import json
os.chdir(COURSE_REPO_DIR)

config_path   = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_cfg_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config   = json.loads(base_cfg_path.read_text())

# Runtime overrides – architecture/eval only, NO timestep keys here
base_config["runtime_overrides"] = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
}

# ── Stage 1: forward-only trot ────────────────────────────────────────────────
base_config["stage_1"]["num_timesteps"]     = 10_000_000     # FIX 1: direct field
base_config["stage_1"]["command_range"]     = {"min": [0.0, 0.0, 0.0], "max": [0.9, 0.0, 0.0]}
base_config["stage_1"]["command_keep_prob"] = [1.0, 0.0, 0.0]

# ── Stage 2: full locomotion with turning ─────────────────────────────────────
base_config["stage_2"]["num_timesteps"]     = 15_000_000     # FIX 1: direct field
base_config["stage_2"]["command_range"]     = {              # FIX 2: include vy + yaw
    "min": [-1.0, -0.4, -1.0],
    "max": [ 1.3,  0.4,  1.0],
}
base_config["stage_2"]["command_keep_prob"] = [0.9, 0.8, 0.8]  # FIX 3: allow vy/yaw
base_config["stage_2"]["reward_scales"]     = {              # FIX 4: add tracking terms
    "tracking_lin_vel": 1.6,
    "tracking_ang_vel": 1.2,
    "feet_air_time":    0.12,
    "action_rate":     -0.015,
    "energy":          -0.0012,
}

config_path.write_text(json.dumps(base_config, indent=2))

# ── Verify (asserts catch mis-configs BEFORE the 3+ hour training) ────────────
cfg = json.loads(config_path.read_text())
s1, s2 = cfg["stage_1"], cfg["stage_2"]

print("Stage 1:")
print("  num_timesteps    :", s1["num_timesteps"])
print("  command_range    :", s1["command_range"])
print("  command_keep_prob:", s1["command_keep_prob"])
print("\nStage 2:")
print("  num_timesteps    :", s2["num_timesteps"])
print("  command_range    :", s2["command_range"])
print("  command_keep_prob:", s2["command_keep_prob"])
print("  reward_scales    :", s2["reward_scales"])

assert s1["num_timesteps"] == 10_000_000,  "stage_1 timesteps wrong"
assert s2["num_timesteps"] == 15_000_000,  "stage_2 timesteps wrong"
assert s2["command_range"]["max"][0] == 1.3,  "stage_2 vx max wrong"
assert s2["command_range"]["max"][1] == 0.4,  "stage_2 vy max is 0 – robot can't turn!"
assert s2["command_range"]["max"][2] == 1.0,  "stage_2 yaw max is 0 – robot can't turn!"
assert s2["command_keep_prob"][1]    == 0.8,  "stage_2 vy keep_prob is 0 – robot can't turn!"
assert s2["command_keep_prob"][2]    == 0.8,  "stage_2 yaw keep_prob is 0 – robot can't turn!"
assert "tracking_lin_vel" in s2["reward_scales"], "stage_2 missing tracking rewards"
print("\nAll assertions passed – config is correct.")

!python train.py --config {config_path} --dry-run

In [ ]:
# ── CELL 6: Train fast low-level policy (~3-4 hours) ─────────────────────────
#
# Stage 1 (10M steps, 0-0.9 m/s): builds stable forward trot
# Stage 2 (15M steps, 0-1.30 m/s): extends speed range
# stage_2 auto-restores from stage_1 (restore_previous_stage_checkpoint=true)
#
# IMMEDIATELY run Cell 7 after this completes to back up to Drive.

TRAIN_OUT = COURSE_REPO_DIR / "artifacts" / "fast_policy"
os.chdir(COURSE_REPO_DIR)

!python train.py \
    --config configs/colab_runtime_config.json \
    --stage both \
    --output-dir {TRAIN_OUT}

CHECKPOINT_DIR = TRAIN_OUT / "best_checkpoint"
print("\nCheckpoint exists:", CHECKPOINT_DIR.exists())
if CHECKPOINT_DIR.exists():
    cfg = json.loads((CHECKPOINT_DIR / "ppo_network_config.json").read_text())
    print("  action_size :", cfg.get("action_size"))
    print("  obs_key     :", cfg.get("network_factory_kwargs", {}).get("policy_obs_key"))

In [ ]:
# ── CELL 7: Save checkpoint to Drive (run immediately after Cell 6!) ──────────
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "fast_policy" / "best_checkpoint"
drive_ckpt     = DRIVE_BACKUP / "fast_policy_checkpoint"

if not CHECKPOINT_DIR.exists():
    print("[error] Checkpoint not found. Did Cell 6 finish successfully?")
else:
    if drive_ckpt.exists():
        shutil.rmtree(drive_ckpt)
    shutil.copytree(str(CHECKPOINT_DIR), str(drive_ckpt))
    print("Saved to Drive:", drive_ckpt)
    print("Files:", [f.name for f in drive_ckpt.iterdir()])

# ── Restore from Drive if session restarted ───────────────────────────────────
# Uncomment the block below if you need to restore after a restart:
#
# CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "fast_policy" / "best_checkpoint"
# drive_ckpt = DRIVE_BACKUP / "fast_policy_checkpoint"
# if not CHECKPOINT_DIR.exists() and drive_ckpt.exists():
#     CHECKPOINT_DIR.parent.mkdir(parents=True, exist_ok=True)
#     shutil.copytree(str(drive_ckpt), str(CHECKPOINT_DIR))
#     print("Restored from Drive.")
# print("Checkpoint exists:", CHECKPOINT_DIR.exists())

In [ ]:
# ── CELL 8: Quick smoke test (10 s, no render) ────────────────────────────────
# Tests the fast policy with the starter planner to verify it doesn't fall.
# If the robot moves forward without falling, the policy is ready for MLP training.

CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "fast_policy" / "best_checkpoint"
PLANNER_CONFIG = COURSE_REPO_DIR / "configs" / "starter_planner.json"
SMOKE_DIR      = COURSE_REPO_DIR / "artifacts" / "smoke_test"

os.chdir(COURSE_REPO_DIR)
!python run_track_bonus.py \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --planner-config {PLANNER_CONFIG} \
    --config configs/colab_runtime_config.json \
    --output-dir {SMOKE_DIR} \
    --entry-name {TEAM_NAME} \
    --duration-seconds 10 \
    --no-render

if (SMOKE_DIR / "results.json").exists():
    r = json.loads((SMOKE_DIR / "results.json").read_text())
    m = r["metrics"]
    print("Smoke test:")
    print(f"  fall        : {m['fall']}")
    print(f"  distance    : {m['valid_distance_m']:.1f} m")
    print(f"  mean_speed  : {m['mean_progress_speed']:.3f} m/s")
    if m['fall']:
        print("[warn] Policy falls with starter planner. MLP training may still fix this.")
    else:
        print("Policy stable. Proceeding to MLP training.")
else:
    print("[error] results.json not found")

In [ ]:
# ── CELL 9: In-process CMA-ES MLP training (~60-120 min) ─────────────────────
#
# BUG FIXES vs previous run:
#   Bug 1: warm-up score was discarded (_ = evaluate()) → planner_weights.npz
#          was overwritten by the first CMA-ES candidate (fell at 0.5m, fit=0.001)
#          FIX: warm-up score now initialises best_score; theta0 saved immediately
#   Bug 2: SIGMA0=0.30 → 771-param perturbations too large → MLP output saturates
#          → robot turns sharply off track in first 1-2 steps (walks 73s off-track)
#          FIX: SIGMA0=0.10 → smaller perturbations, more stable initial candidates
#
# Fitness: distance/200 + speed bonus when lap completes.
# fitness > 1.0  → lap completed
# fitness ≈ 1.5  → lap in ~175 s (excellent)

import numpy as np, time, json

if str(COURSE_REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(COURSE_REPO_DIR.resolve()))

from track_bonus.planner import MLPTrackPlanner, _VX_MAX, _VX_MIN, _VY_LIM, _YAW_LIM
from track_bonus.official_track import official_track
from track_bonus.scoring import compute_track_bonus_metrics
from course_common import lazy_import_stack, load_json, set_runtime_env
from run_track_bonus import rollout, _make_env
from test_policy import load_policy_with_workaround

CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "fast_policy" / "best_checkpoint"
HIGHLEVEL_DIR  = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
HIGHLEVEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Speed limits: VX=[{_VX_MIN}, {_VX_MAX}]  VY=±{_VY_LIM}  YAW=±{_YAW_LIM}")
expected_time = 114.7/(0.8*_VX_MAX) + 85.3/_VX_MAX
print(f"Expected lap time (80% curve speed): {expected_time:.1f} s")

# ── Hyper-params ──────────────────────────────────────────────────────────────
HIDDEN_SIZES  = [32, 16]
N_GENERATIONS = 20       # was 10 (more time for convergence)
POPULATION    = 8
SIGMA0        = 0.10     # was 0.30 → FIX: smaller perturbations, robot stays on track
SEED          = 42
EVAL_SECONDS  = 280.0

# ── Setup ONCE ────────────────────────────────────────────────────────────────
print("\nLoading env + policy...")
set_runtime_env()
course_cfg = load_json(COURSE_REPO_DIR / "configs" / "colab_runtime_config.json")
course_cfg["runtime_overrides"] = {}
num_steps = int(round(EVAL_SECONDS / course_cfg["control"]["ctrl_dt"]))
track  = official_track()
stack  = lazy_import_stack()
env    = _make_env(stack, course_cfg, "stage_2", num_steps)
policy = load_policy_with_workaround(CHECKPOINT_DIR.resolve(), deterministic=True)
policy = stack["jax"].jit(policy)
print(f"Ready. EVAL_SECONDS={EVAL_SECONDS}  num_steps={num_steps}")

# ── Fitness ───────────────────────────────────────────────────────────────────
def evaluate(theta, seed=SEED):
    weights = MLPTrackPlanner.unpack(theta, HIDDEN_SIZES)
    planner = MLPTrackPlanner(weights, HIDDEN_SIZES, stand_seconds=1.0)
    result  = rollout(
        stack=stack, env=env, policy=policy, planner=planner,
        track=track, num_steps=num_steps, seed=seed, start_s=0.0, force_cpu=False,
    )
    m    = compute_track_bonus_metrics(result, track)
    dist = m["valid_distance_m"] / 200.0
    fall = m.get("fall", True)
    ft   = m.get("finish_time")
    fit  = min(dist, 1.0)
    if ft is not None:
        fit = 1.0 + max(0.0, (280.0 - float(ft)) / 280.0)
    if fall:
        fit *= 0.55
    return float(fit), m

# ── CMA-ES (diagonal covariance) ──────────────────────────────────────────────
class _CMAes:
    def __init__(self, x0, sigma0=0.1, popsize=8, seed=0):
        self.rng = np.random.default_rng(seed)
        self.n   = len(x0)
        self.mean  = x0.copy().astype(np.float64)
        self.sigma = float(sigma0)
        self.lam   = popsize
        self.mu    = max(popsize // 2, 2)
        raw_w = np.log(self.mu + 0.5) - np.log(np.arange(1, self.mu + 1))
        self.w = raw_w / raw_w.sum()
        self.mueff = 1.0 / float(np.sum(self.w ** 2))
        self.cs = (self.mueff + 2.0) / (self.n + self.mueff + 5.0)
        self.ds = 1.0 + 2.0 * max(0.0, np.sqrt((self.mueff-1.0)/(self.n+1.0))-1.0) + self.cs
        self.chiN = float(np.sqrt(self.n)*(1.0-1.0/(4.0*self.n)+1.0/(21.0*self.n**2)))
        self.ps  = np.zeros(self.n)
        self.var = np.ones(self.n)

    def ask(self):
        return self.mean + self.sigma * np.sqrt(self.var) * self.rng.standard_normal((self.lam, self.n))

    def tell(self, xs, scores):
        order     = np.argsort(-scores)
        elite     = xs[order[:self.mu]]
        old_mean  = self.mean.copy()
        self.mean = (self.w[:, None] * elite).sum(axis=0)
        step = (self.mean - old_mean) / (self.sigma * np.sqrt(self.var) + 1e-12)
        self.ps = (1-self.cs)*self.ps + np.sqrt(self.cs*(2-self.cs)*self.mueff)*step
        self.sigma *= float(np.exp((self.cs/self.ds)*(np.linalg.norm(self.ps)/self.chiN-1.0)))
        self.sigma  = float(np.clip(self.sigma, 1e-8, 2.0))
        ys = (elite - old_mean) / (self.sigma * np.sqrt(self.var) + 1e-12)
        self.var = np.clip(0.9*self.var + 0.1*float(np.sum(self.w))*(self.w[:,None]*ys**2).sum(0), 1e-10, None)

# ── Init ──────────────────────────────────────────────────────────────────────
n_params = MLPTrackPlanner.param_count(HIDDEN_SIZES)
print(f"\nMLP 5→{HIDDEN_SIZES}→3  ({n_params} params)")
print(f"CMA-ES: {N_GENERATIONS} gens × {POPULATION} pop  sigma={SIGMA0}  eval={EVAL_SECONDS}s")

theta0 = MLPTrackPlanner.pack(MLPTrackPlanner.make_weights(HIDDEN_SIZES, seed=SEED))

print("\nWarm-up rollout (JAX JIT, ~2-3 min)...")
t0 = time.time()
warmup_fit, wm = evaluate(theta0, seed=SEED)   # FIX: capture score (was discarded with _)
print(f"Warm-up done in {time.time()-t0:.1f}s")
print(f"  fit={warmup_fit:.4f}  distance={wm['valid_distance_m']:.1f}m  fall={wm.get('fall')}  finish_time={wm.get('finish_time')}")

# FIX: initialise best from warm-up so CMA-ES candidates must BEAT theta0 to overwrite
best_score = warmup_fit
best_theta = theta0.copy()
np.savez(str(HIGHLEVEL_DIR / "planner_weights.npz"),
         **MLPTrackPlanner.unpack(best_theta, HIDDEN_SIZES))
print(f"theta0 saved as initial best: fit={best_score:.4f}  ({wm['valid_distance_m']:.1f}m)")

es      = _CMAes(theta0, sigma0=SIGMA0, popsize=POPULATION, seed=SEED)
history = []

# ── Loop ──────────────────────────────────────────────────────────────────────
for gen in range(N_GENERATIONS):
    t_gen      = time.time()
    candidates = es.ask()
    scores     = np.zeros(POPULATION)
    print(f"\n── Gen {gen+1}/{N_GENERATIONS}  σ={es.sigma:.4f} ──")

    for idx, theta in enumerate(candidates):
        t0 = time.time()
        score, m = evaluate(theta, seed=SEED + gen*100 + idx)
        scores[idx] = score
        ft  = m.get("finish_time")
        tag = f"LAP {ft:.1f}s" if ft is not None else f"dist={m['valid_distance_m']:.1f}m"
        star = "★" if score > best_score else " "
        print(f"  {star} [{idx+1}/{POPULATION}] fit={score:.4f}  {tag}  fall={m.get('fall')}  ({time.time()-t0:.1f}s)")
        if score > best_score:
            best_score = score
            best_theta = theta.copy()
            np.savez(str(HIGHLEVEL_DIR / "planner_weights.npz"),
                     **MLPTrackPlanner.unpack(best_theta, HIDDEN_SIZES))

    es.tell(candidates, scores)
    gen_t = time.time() - t_gen
    history.append({"gen": gen, "best": float(best_score),
                    "max": float(scores.max()), "mean": float(scores.mean()),
                    "sigma": float(es.sigma), "time_s": gen_t})
    print(f"  Best={best_score:.4f}  GenMax={scores.max():.4f}  ({gen_t:.0f}s)")
    (HIGHLEVEL_DIR / "search_history.json").write_text(json.dumps(history, indent=2))

# ── Save final config ─────────────────────────────────────────────────────────
final_cfg = {
    "planner_type":     "mlp",
    "mlp_weights_path": "planner_weights.npz",
    "mlp_hidden":       HIDDEN_SIZES,
    "stand_seconds":    1.0,
    "training_info":    {
        "vx_max": _VX_MAX, "vx_min": _VX_MIN,
        "vy_lim": _VY_LIM, "yaw_lim": _YAW_LIM,
        "best_fitness": float(best_score),
        "generations": N_GENERATIONS, "population": POPULATION,
        "sigma0": SIGMA0,
        "target": "200m in <200s",
    },
}
(HIGHLEVEL_DIR / "planner_config.json").write_text(json.dumps(final_cfg, indent=2))
print(f"\nDone. Best fitness={best_score:.4f}")
print("  >1.0 = lap completed; ≈0.15 = 30m walk; ≈0.43 = 85m (one straight)")

In [ ]:
# ── CELL 10: Save MLP to Drive ────────────────────────────────────────────────
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
drive_mlp     = DRIVE_BACKUP / "highlevel_mlp"

if drive_mlp.exists():
    shutil.rmtree(drive_mlp)
shutil.copytree(str(HIGHLEVEL_DIR), str(drive_mlp))
print("Saved to Drive:", drive_mlp)
for f in drive_mlp.iterdir():
    print(" ", f.name)

# ── Restore if restarted ──────────────────────────────────────────────────────
# Uncomment if you need to restore after session restart:
# HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
# if not HIGHLEVEL_DIR.exists() and (DRIVE_BACKUP / "highlevel_mlp").exists():
#     shutil.copytree(str(DRIVE_BACKUP / "highlevel_mlp"), str(HIGHLEVEL_DIR))
#     print("Restored from Drive.")

In [ ]:
# ── CELL 11: Full track evaluation (300 s, with video) ───────────────────────
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "fast_policy" / "best_checkpoint"
PLANNER_CONFIG = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp" / "planner_config.json"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

os.chdir(COURSE_REPO_DIR)
!python run_track_bonus.py \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --planner-config {PLANNER_CONFIG} \
    --config configs/colab_runtime_config.json \
    --output-dir {TRACK_EVAL_DIR} \
    --entry-name {TEAM_NAME} \
    --duration-seconds 300 \
    --render-every 10 \
    --render-fps 5

if (TRACK_EVAL_DIR / "results.json").exists():
    r = json.loads((TRACK_EVAL_DIR / "results.json").read_text())
    m, s = r["metrics"], r["scores"]
    print("\n=== RESULTS ===")
    print(f"  composite_score  : {s['composite_score']:.4f}")
    print(f"  lap_completion   : {m['lap_completion']}")
    print(f"  finish_time      : {m['finish_time']}  (target: <200s)")
    print(f"  mean_speed       : {m['mean_progress_speed']:.3f} m/s  (target: >1.0)")
    print(f"  fall             : {m['fall']}")
    print(f"  rms_lateral_err  : {m['rms_lateral_error']:.3f} m")
    ft = m['finish_time']
    if ft is not None and float(ft) < 200:
        print(f"\n  TARGET MET: {ft:.1f}s < 200s")
    elif ft is not None:
        print(f"\n  Close: {ft:.1f}s (need more MLP training or higher VX_MAX)")
    else:
        print("\n  Lap not completed in 300s – increase N_GENERATIONS in Cell 9 and retrain MLP")
else:
    print("[error] results.json not found")

In [ ]:
# ── CELL 12: submission.json + copy artifacts + final checklist ───────────────
import json

HIGHLEVEL_DIR  = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "fast_policy" / "best_checkpoint"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

# Create submission.json
submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz",
    "high_level_planner_type": "learned_mlp_cmaes",
    "track_eval": "track_eval/results.json",
    "notes": (
        "Low-level: Brax PPO trained from scratch. "
        "Stage 1: 10M steps, vx 0-0.9 m/s (stable trot foundation). "
        "Stage 2: 15M steps, vx 0-1.30 m/s (extended speed for sub-200s target). "
        "High-level: 5->32->16->3 MLP (VX_MAX=1.15, VY=0.25, YAW=0.50) trained with "
        "diagonal CMA-ES (10 gen x 8 pop), in-process evaluation (single JAX JIT). "
        "Fitness = dist/200 + speed_bonus if lap completed. "
        "Failed attempts: train from scratch at 1.2 m/s in stage_1 (no stable gait), "
        "subprocess CMA-ES (120s JIT per candidate), fine-tune via --restore-checkpoint-dir "
        "(exported best_checkpoint format incompatible)."
    ),
}
(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))

# Copy artifacts to repo root
shutil.copy(str(HIGHLEVEL_DIR / "planner_config.json"), str(COURSE_REPO_DIR / "planner_config.json"))
shutil.copy(str(HIGHLEVEL_DIR / "planner_weights.npz"), str(COURSE_REPO_DIR / "planner_weights.npz"))

dest_ckpt = COURSE_REPO_DIR / "best_checkpoint"
if dest_ckpt.exists():
    shutil.rmtree(dest_ckpt)
shutil.copytree(str(CHECKPOINT_DIR), str(dest_ckpt))

dest_eval = COURSE_REPO_DIR / "track_eval"
if dest_eval.exists():
    shutil.rmtree(dest_eval)
shutil.copytree(str(TRACK_EVAL_DIR), str(dest_eval))

# Checklist
expected = {
    "best_checkpoint/"        : COURSE_REPO_DIR / "best_checkpoint",
    "planner_config.json"     : COURSE_REPO_DIR / "planner_config.json",
    "planner_weights.npz"     : COURSE_REPO_DIR / "planner_weights.npz",
    "track_bonus/planner.py"  : COURSE_REPO_DIR / "track_bonus" / "planner.py",
    "track_eval/results.json" : COURSE_REPO_DIR / "track_eval" / "results.json",
    "submission.json"         : COURSE_REPO_DIR / "submission.json",
}
all_ok = True
print("=== SUBMISSION CHECKLIST ===")
for label, path in expected.items():
    ok = path.exists()
    if not ok:
        all_ok = False
    print("OK  " if ok else "MISS", label)

if all_ok:
    drive_final = DRIVE_BACKUP / "final_submission"
    if drive_final.exists():
        shutil.rmtree(drive_final)
    drive_final.mkdir(parents=True)
    for label, path in expected.items():
        dest = drive_final / label.rstrip("/")
        dest.parent.mkdir(parents=True, exist_ok=True)
        if path.is_dir():
            shutil.copytree(str(path), str(dest))
        else:
            shutil.copy(str(path), str(dest))
    print(f"\nAll OK. Packaged to Drive: {drive_final}")

    results_path = COURSE_REPO_DIR / "track_eval" / "results.json"
    if results_path.exists():
        r = json.loads(results_path.read_text())
        print(f"\ncomposite_score : {r['scores']['composite_score']:.4f}")
        print(f"finish_time     : {r['metrics']['finish_time']}  (target <200s)")
        print(f"fall            : {r['metrics']['fall']}")

In [ ]:
# ── CELL 13: Push to GitHub ───────────────────────────────────────────────────
import getpass
os.chdir(COURSE_REPO_DIR)

!git config --global user.email "jrzzhang@ucdavis.edu"
!git config --global user.name "Jiarao Zhang"

# Stage all submission artifacts (force to bypass .gitignore)
!git add -f submission.json track_bonus/planner.py planner_config.json planner_weights.npz
!git add -f best_checkpoint/ track_eval/
!git status --short

token = getpass.getpass("GitHub token (repo scope): ")
REMOTE = f"https://{token}@github.com/jiarao76/Final-Project-Track-2-Bonus-Project.git"

!git commit -m "Add fast policy (stage2 1.30 m/s), MLP planner, and track eval results"
!git push {REMOTE} main
print("Pushed to GitHub.")